In [1]:
l = [0, 2, 1, 3, 5, 2, 1]
sorted([(_, i) for _, i in enumerate(l)])

[(0, 0), (1, 2), (2, 1), (3, 3), (4, 5), (5, 2), (6, 1)]

In [2]:
# brew install ola
# conda install protobuf

# https://www.enttec.com/eu/products/controls/dmx-usb/dmx-usb-pro/

# 1/ http://www.ftdichip.com/Drivers/D2XX.htm : install D2xxHelper (in /Library/Extensions/)
# 2/ sudo kext unload -b com.apple.driver.AppleUSBFTDI
# 3/ https://www.enttec.com/product/pro-manager/ : download + start
# -> Pro Manager : select device from dropdown; showing firmware 1.44, EN237452
# DMX Send : From Faders (see channels below; e.g. 1, 6) -> green LED starts blinking
# Note: Froggy LED must be set into "d001" mode (11 channels, starting at 1)
# (LED continues blinking after pro manager exits until cable disconnected)

# installed http://www.ftdichip.com/Drivers/D2XX/MacOSX/D2XX1.4.4.dmg
# ... this let's clients use FT_ library calls to access via libusb

# http://www.ftdichip.com/Drivers/VCP/MacOSX/FTDIUSBSerialDriver_v2_4_2.dmg
# sudo kextload -b com.apple.driver.AppleUSBFTDI
# -> /dev/tty.usbserial-EN237452 when device is connected

# Ubuntu 17.04 Setup
# sudo apt-get install -y ola-python ola
# sudo mv /etc/ola/ola-stageprofi.conf ~/Documents/
# 

# also described here :)
# https://www.openlighting.org/ola/getting-started/device-specific-configuration/#Enttec_USB_Pro

# echo > ~/.ola/ola-usbpro.conf <<EOC
#device_dir = /dev
#device_prefix = tty.usbserial-
#EOC

# olad -l 2

# then create universe "1" at http://localhost:9090/ola.html specify ENTTEC output
# play around with DMX Console

In [4]:
d = {}
1 if d else 0

0

In [14]:
paths = [
#     '/usr/local/Cellar/protobuf/3.5.1_1/libexec/lib/python2.7/site-packages',
    '/usr/local/Cellar/ola/0.10.6/lib/python2.7/site-packages',
    '/usr/lib/python2.7/dist-packages/'
]
import sys
for path in paths:
    if path not in sys.path:
        sys.path.append(path)

In [15]:
import array
from ola.ClientWrapper import ClientWrapper

In [16]:
import sys
print(sys.path)

['/home/gabriel/git/rizhom/env/lib/python36.zip', '/home/gabriel/git/rizhom/env/lib/python3.6', '/home/gabriel/git/rizhom/env/lib/python3.6/lib-dynload', '/usr/lib/python3.6', '', '/home/gabriel/git/rizhom/env/local/lib/python3.6/site-packages', '/home/gabriel/git/rizhom/env/lib/python3.6/site-packages', '/home/gabriel/git/rizhom/env/local/lib/python3.6/site-packages/IPython/extensions', '/home/gabriel/.ipython', '/usr/local/Cellar/ola/0.10.6/lib/python2.7/site-packages', '/usr/lib/python2.7/dist-packages/']


# Animations

In [7]:
# https://cdn.competec.ch/documents/3/5/353589/Manual.pdf
# 1 - 6 : dimmer, strobe, rgbw
# 7 - 9 : pan, tilt, speed
#  pan : 0=148: 0
#        37=184: pi/2
#        72=221: pi
#        109=255: 3pi/2
#  tilt: 38: 0
#        86: pi/4
#        133: pi/2
#        191: 3pi/4
#        229: pi

In [44]:
wrapper = ClientWrapper()
client = wrapper.Client()

def set_channels(a):
    a = array.array('B', map(int, a))
    client.SendDmx(1, a, lambda state: wrapper.Stop())

In [47]:
# Red blue left right
# start position pi
a = [255, 0, 0, 255, 0, 255, 0, 255, 0]
set_channels(a)

/home/gabriel/git/rizhom/env/local/lib/python3.6/site-packages/ipykernel_launcher.py:6: DeprecationWarning: tostring() is deprecated. Use tobytes() instead.
  


In [11]:
# start position pi
a = [255, 0, 0, 0, 0, 255, 72, 133, 0]
set_channels(a)

/home/gabriel/git/rizhom/env/local/lib/python3.6/site-packages/ipykernel_launcher.py:6: DeprecationWarning: tostring() is deprecated. Use tobytes() instead.
  


In [11]:
# pan 2pi forth + back while tilting downwards
import time
a[8] = 0  # 255 -> crash
n = 1  # times forth and back
steps = 6  # intermediate steps
turn_secs = .9  # empirical
p0, p1 = 221, 72
dp = (p1 - p0) / (steps + 1)
pans = [p0] + [p0 + (i + 1)*dp for i in range(steps)] + [p1] + [p1 - (i + 1)*dp for i in range(steps)]
i_n = n * len(pans) + 1
for i in range(i_n):
    a[6] = pans[i % len(pans)]
    a[7] -= (133-38) / i_n
    set_channels(a)
    time.sleep(turn_secs / (steps + 1))

/home/gabriel/git/rizhom/env/lib/python3.6/site-packages/ipykernel_launcher.py:6: DeprecationWarning: tostring() is deprecated. Use tobytes() instead.
  


# Initial experiments

In [17]:
# https://www.openlighting.org/ola/developer-documentation/python-api/
# http://localhost:9090/ola.html

universe = 1
data = array.array('B', [255, 0, 0, 0, 0, 255])
wrapper = ClientWrapper()
client = wrapper.Client()

def DmxSent(state):
    wrapper.Stop()  # or wrapper will block at next call

client.SendDmx(universe, data, DmxSent)
wrapper.Run()

In [71]:

wrapper = None
loop_count = 0
TICK_INTERVAL = 100  # in ms
SECONDS = 10

def StopIfFail(state):
    if not state.Succeeded():
        wrapper.Stop()
def StopWhenDone(state):
    wrapper.Stop()

def SendDMXFrame():
    global loop_count
    if loop_count < 1000 * SECONDS / TICK_INTERVAL:
        wrapper.AddEvent(TICK_INTERVAL, SendDMXFrame)
        sent = StopIfFail
    else:
        sent = StopWhenDone

    data = array.array('B', [255, 0, 0, 0, 0, 255, (loop_count * 2) % 150, 103])
    loop_count += 1

    wrapper.Client().SendDmx(1, data, sent)

wrapper = ClientWrapper()
wrapper.AddEvent(TICK_INTERVAL, SendDMXFrame)
wrapper.Run()

In [ ]:
wrapper.Stop()

In [75]:

frames = [
    (0,  [255, 0, 0, 0, 0, 0, 155, 15]), # table
    (5,  [255, 0, 0, 0, 0, 0, 155, 15]), # table
    (6,  [255, 0, 255, 0, 0, 0, 155, 15]), # table
    (8,  [255, 0, 255, 0, 0, 0, 155, 47]), # L
    (10, [255, 0, 255, 0, 0, 0, 147, 50]), # E
    (15, [255, 0,   0, 0, 0, 0, 147, 50]), # dark
]

# circle top
frames = [
    (0,  [255, 0, 0, 0, 0, 255, 0, 103]),
    (1,  [255, 0, 0, 0, 0, 255, 150, 103]),
    (2,  [255, 0, 0, 0, 0, 255, 0, 103]),
    (3,  [255, 0, 0, 0, 0, 255, 140, 103]),
]

wrapper = None
loop_count = 0
TICK = 5  # in ms

def StopIfFail(state):
    if not state.Succeeded():
        wrapper.Stop()
def StopWhenDone(state):
    wrapper.Stop()

def SendDMXFrame():
    global loop_count
    s = loop_count * TICK / 1000.
    done = False
    for i, (when, frame) in enumerate(frames):
        if s >= when:
            if i == len(frames) - 1:
                done = True
            else:
                when2, frame2 = frames[i + 1]
                x = (s - when) / (when2 - when)
                data = [int(frame[j] + x * (frame2[j] - frame[j]))
                        for j in range(len(frame))]

    if not done:
        wrapper.AddEvent(TICK, SendDMXFrame)
        sent = StopIfFail
    else:
        sent = StopWhenDone

    data = array.array('B', data)
    loop_count += 1

    wrapper.Client().SendDmx(1, data, sent)

wrapper = ClientWrapper()
wrapper.AddEvent(TICK, SendDMXFrame)
wrapper.Run()